# Session 6 · Language as a Service
*Language Machines: How AI Writes Culture*

**In this session you will:** watch a machine produce the "commonplaces" of your sector's paperwork, measure how much of a machine draft you actually change when you edit it, map your own job into what you would and would not hand over, and close the course by debating Weatherby's argument as a whole.

**Time:** about 90 minutes. Please use *Runtime → Change runtime type → T4 GPU*.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

In software, things are increasingly sold "as a service": storage, music, even jet engines, rented rather than owned. Weatherby's conclusion is that **language itself is now on offer as a service**, available on tap, by the paragraph.

He does not predict an apocalypse. He predicts something more banal and perhaps more lasting. When the Industrial Revolution arrived, the work of calculation was split: the "manual" labour of computing was separated from mathematical thinking, and eventually handed to machines. He argues writing is now being split the same way, into automated **labour** and human direction.

Automation rarely replaces people cleanly. It creates new, lesser work around the machine. His image is the worker who used to operate one loom and now patrols a floor of machines, or the attendant who supervises six self-checkout tills at once. He expects many writing jobs (and coding jobs) to become this kind of **attending**.

His answer is the **return of rhetoric**. For nearly two thousand years, rhetoric was the core of education: training in composition, memory, persuasion and the **commonplaces** (stock phrases and arguments everyone drew on). Machines are now producing a new set of commonplaces at scale. Learning to recognise, use and resist them may become a basic cultural skill again.

## Setup

In [ ]:
#@title Setup: load a small chat model (takes 1–3 minutes the first time)
# A small, openly available chat model. It is far weaker than ChatGPT or Claude,
# which is useful: its habits and defaults are easier to see.
import torch, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(device=device, dtype=dtype)

def ask(prompt, system="You are a helpful assistant.", temperature=0.8,
        max_new_tokens=250, show=True):
    """Send one message to the chat model and return its reply."""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = chat_tok(text, return_tensors="pt").to(device)
    out = chat_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperature, top_p=0.9,
                              pad_token_id=chat_tok.eos_token_id)
    reply = chat_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show:
        for para in reply.split("\n"):
            print(textwrap.fill(para, 90) if para.strip() else "")
    return reply

print(f"Chat model ready (running on {device}).")

In [ ]:
#@title Setup: tools for finding repeated phrases (instant)
import re
from collections import Counter

STOP = set("""a an the and or but of to in on at for with by from as is are was were be been it its this that
these those i you he she we they my your our their his her me us them not no so if then than""".split())

def common_phrases(texts, sizes=(2, 3, 4), min_texts=3, top=20):
    """Find phrases that appear in at least `min_texts` different texts."""
    seen = Counter()
    for t in texts:
        words = re.findall(r"[a-z']+", t.lower())
        grams = set()
        for n in sizes:
            for i in range(len(words) - n + 1):
                g = words[i:i+n]
                if all(w in STOP for w in g):
                    continue
                grams.add(" ".join(g))
        seen.update(grams)
    rows = [(g, c) for g, c in seen.most_common() if c >= min_texts]
    # hide short phrases that only ever appear inside a longer repeated phrase
    rows = [(g, c) for g, c in rows
            if not any(g != h and f" {g} " in f" {h} " and c == d for h, d in rows)][:top]
    if not rows:
        print("No phrase appeared in that many texts. Try lowering min_texts.")
    for g, c in rows:
        print(f"{c:>3} texts  ·  {g}")
    return rows

def word_counts(texts, top=25):
    words = [w for t in texts for w in re.findall(r"[a-z']+", t.lower()) if w not in STOP and len(w) > 2]
    return Counter(words).most_common(top)

print("Ready.")

## Part 1 · The new commonplaces

We ask the model for the opening paragraph of a funding application eight times, for the same (fictional) project. Then we look for the stock phrases they share. Change the project description to one of your own if you like.

In [ ]:
project = ("a six-month community oral-history project recording the memories of retired "
           "dockworkers and market traders, ending in a public exhibition and a podcast")

openings = []
for i in range(8):
    print(f"\n--- version {i+1} ---")
    openings.append(ask(f"Write the opening paragraph of a funding application for {project}.",
                        max_new_tokens=150))

In [ ]:
print("THE NEW COMMONPLACES:\n")
common_phrases(openings, min_texts=3)

**Discussion:** funders read hundreds of applications. What happens when many of them are drafted from the same commonplaces? Does it help people who struggle with the genre get through the door, or does it flatten everyone into the same voice? Both?

## Part 2 · The attendant's work

Here we measure the division of labour directly. The model writes a draft; you edit it until you would be willing to put your name to it; then we calculate how much of the final text is yours.

In [ ]:
draft = ask("Write a 60-word description of a jazz night at a small gallery, for social media.",
            max_new_tokens=120)

Copy the draft into the box below between the triple quotes, then **edit it** into something you would actually post. Then run the box.

In [ ]:
my_version = """Paste the draft here and edit it until it sounds like you."""

import difflib
ratio = difflib.SequenceMatcher(None, draft.split(), my_version.split()).ratio()
print(f"About {ratio*100:.0f}% of your final text overlaps with the machine's draft.")
print(f"About {100 - ratio*100:.0f}% is yours.\n")
for token in difflib.ndiff(draft.split(), my_version.split()):
    if token.startswith("- "): print("  removed:", token[2:])
    elif token.startswith("+ "): print("  added:  ", token[2:])

**Reflect:** what kinds of words did you remove? What did you add? Many people find they remove adjectives and generic enthusiasm, and add specifics: names, places, times, a real detail only someone who was there would know. That pattern says a lot about where the human part of writing lives.

## Part 3 · Map your own work

Fill in the table with writing tasks from your own job. For each, decide whether it is mostly **labour** (it needs doing, and doing adequately) or mostly **judgement** (the value is in *what* is said and by whom), and whether you would hand it to a machine.

Edit the rows, keeping the same format, then run the box.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

my_tasks = [
    # (task,                               labour or judgement?, would you delegate? yes / partly / no, why)
    ("Grant application: budget narrative",  "labour",    "partly", "Structure is standard, numbers must be mine"),
    ("Artist statement",                      "judgement", "no",     "It is the artist's voice"),
    ("Social media event posts",              "labour",    "yes",    "Repetitive, low stakes"),
    ("Wall texts for an exhibition",          "judgement", "partly", "First draft maybe, final voice is curatorial"),
    ("Reply to a complaint from a visitor",   "judgement", "no",     "Needs care and accountability"),
    ("Translating a press release",           "labour",    "partly", "Depends on the language, see Session 5"),
]

df = pd.DataFrame(my_tasks, columns=["task", "kind", "delegate?", "why"])
display(df)
df.groupby(["kind", "delegate?"]).size().unstack(fill_value=0).plot.bar(
    stacked=True, rot=0, figsize=(6, 4), title="My writing work")
plt.ylabel("number of tasks"); plt.show()

Share your table with a partner. Where did you disagree? Weatherby's prediction is that the boundary between labour and judgement will keep moving, and that organisations will be tempted to redraw it for cost reasons rather than cultural ones. Who in your organisation should decide where it sits?

## Part 4 · Closing debate: is Weatherby right?

The book is a polemic, and a good course should argue with it. Split into two groups and prepare the strongest case for each side.

**Weatherby's side.** These machines show that language is a cultural system that can run without a mind. Insisting on a special "human" core that machines can never touch (what he calls **remainder humanism**) is a losing game: the line keeps moving, and it distracts from studying what these systems actually do to culture, power and work. The humanities should stop defending the border and start analysing the machine, with tools from structuralism, poetics and rhetoric.

**The critics' side** (drawing on Emily Bender, Timnit Gebru, Noam Chomsky and others). Meaning requires someone who means it: an intention, a body, a community, accountability. Text without that is a simulation of meaning, and treating it as language risks handing cultural authority to companies whose incentives are commercial. Focusing on the "human" is not sentimental; it is how we insist on responsibility for what gets said.

**Questions for the debate:**
1. When a chatbot at a museum describes a painting, who is responsible for what it says?
2. Does it matter whether machine text "really" means something, if people are moved, informed or misled by it?
3. What would a cultural sector look like that neither panics about AI nor surrenders to it?

## Capstone options

1. **Ideology scan report.** Using the tools from Session 5, investigate the defaults a model holds about a topic you care about, and present your findings.
2. **Made with, and about, the machine.** Create a short piece (text, performance script, sound or visual work) with a model, plus a critical commentary on the process using ideas from the course.
3. **An AI policy for your organisation.** A two-page draft covering where machine-written language may and may not be used, how it is disclosed, and who decides.

## Going further
- Weatherby, *Language Machines*, Conclusion.
- Matthew Kirschenbaum, "Prepare for the Textpocalypse", *The Atlantic* (2023).
- Ted Chiang, "ChatGPT Is a Blurry JPEG of the Web", *The New Yorker* (2023).
- Emily Bender and Alexander Koller, "Climbing towards NLU" (2020), the "octopus" thought experiment.